In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

loans = pd.read_csv('/data/gold_loans.csv')
branch = pd.read_csv('/data/branches.csv')
repayment = pd.read_csv('/data/repayments.csv')
customers = pd.read_csv('/data/customers.csv')
gold_rates = pd.read_csv('/data/gold_rates.csv')

loans.head()
loans.tail()

In [ ]:
tables = {
    'loans': loans,
    'branch': branch,
    'repayment': repayment,
    'customers': customers,
    'gold_rates': gold_rates
}

def clean_table(df):
    df = df.copy()
    # 1. Standardize column headers (lowercase, no spaces)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
    # 2. Trim whitespace from text columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
        
    # 3. Replace fake null strings with actual NaN
    df = df.replace(['', 'NULL', 'null', 'N/A', 'NA', 'None', 'nan', '-'], np.nan)
    
    # 4. Drop exact duplicate rows
    df = df.drop_duplicates()
    return df

for name in tables:
    tables[name] = clean_table(tables[name])
    print(f"=== {name.upper()} | Shape: {tables[name].shape} ===")
    print("Columns:", tables[name].columns.tolist())
    print("Missing values:\n", tables[name].isna().sum()[tables[name].isna().sum() > 0], "\n")

# Unpack back into your original variable names
loans = tables['loans']
branch = tables['branch']
repayment = tables['repayment']
customers = tables['customers']
gold_rates = tables['gold_rates']


In [ ]:
# 1. Clean join/primary key IDs (update column names if Stage 1 prints slightly different names)
loans['loan_id'] = loans['loan_id'].astype(str).str.upper().str.strip()

# 2. Identify the exact gold weight column name automatically
weight_col = [c for c in loans.columns if 'weight' in c][0]
print("Detected weight column:", weight_col)

# 3. Convert gold weight to numeric (turns text/errors into NaN)
loans[weight_col] = pd.to_numeric(loans[weight_col], errors='coerce')

# 4. Deduplicate by loan_id (sort first so valid weights are kept over NaN rows)
loans = loans.sort_values(by=weight_col, na_position='last').drop_duplicates(subset=['loan_id'], keep='first')


In [ ]:
null_weights = loans[weight_col].isna().sum()
zero_or_neg = (loans[weight_col] <= 0).sum()

print("Unique loans count:", len(loans))
print("Loans with NaN/null gold weight:", null_weights)
print("Loans with <= 0 gold weight:", zero_or_neg)
print("Total loans with no valid gold weight:", null_weights + zero_or_neg)


In [ ]:
# How many rows are in gold_loans exactly as received (before any cleaning)?

In [ ]:
loans.shape

In [ ]:
# How many rows are in repayments?

In [ ]:
repayment.shape

In [ ]:
# How many branches does Suvarna Finance have?

In [ ]:
branch['branch_id'].nunique()

In [ ]:
# In how many different states does the company have branches?

In [ ]:
branch['state'].nunique()


In [ ]:
# Return one row with the earliest and latest disbursal dates as first_disbursal and last_disbursal.

In [ ]:

dates=pd.to_datetime(loans['disbursal_date'])
answer=pd.DataFrame({
    'first_disbursal':[dates.min()],
    'last_disbursal':[dates.max()]
}
)


In [ ]:
# How many distinct loan_id values are there?

In [ ]:
loans['loan_id'].nunique()


In [ ]:
# How many exact duplicate rows must be removed from gold_loans?

In [ ]:
loans.duplicated().sum()

In [ ]:
# In how many different ways is status written in the raw data (count distinct raw values, letter case included)?

In [ ]:
loans['status'].nunique()

In [ ]:
#After removing duplicates and converting status to lower case, return status and loans (the number of loans) for each status.

loans.shape

clean_loans=loans.drop_duplicates().copy()
clean_loans.shape
clean_loans['status']=clean_loans['status'].str.strip().str.lower()
answer=clean_loans['status'].value_counts().reset_index(name='Loans')

In [ ]:
# How many different raw values does the purity column contain?

In [ ]:
clean_loans['purity'].unique()
import numpy as np

arr = np.array(['22K', '999', '22KT', '916', '20K', '18K', '24', '750', '22','18', '24K','833', '20'], dtype=object)

In [ ]:
#After removing duplicates and converting purity to a karat number (916 = 22, 833 = 20, 750 = 18, 999 = 24),
 #return karat and loans for each karat

karat={'916':22,'833':20,'750':18,'999':24}

def to_karat(p):
    num=''.join(i for i in str(p) if i.isdigit())
    return karat.get(num,int(num))

clean_loans['purity']=clean_loans['purity'].apply(to_karat)



In [ ]:
# After removing duplicates, how many loans have no gold weight recorded?



In [ ]:
df_clean = df.drop_duplicates(subset=['loan_id']) 
missing_weight_count = df_clean['loan_id'].isna().sum() 
print("Loans with no gold weight recorded:", missing_weight_count)

In [ ]:
print(df.columns.tolist())

In [ ]:
# After removing duplicates, how many loans have a principal of 0 or less?

In [ ]:


# 1. Print all matching columns to see which ones exist
matches = [c for c in loans.columns if 'principal' in c or 'amount' in c]
print("Matching columns:", matches)

# 2. Pick the first match (or replace matches[0] with the exact column name, e.g., 'principal_amount')
col = matches[0]

# 3. Convert that single column to numeric safely
loans[col] = (
    loans[col]
    .astype(str)
    .str.replace(r'[^\d.-]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)

# 4. Remove duplicates and count principal <= 0
loans_clean = loans.drop_duplicates(subset=['loan_id'])
print("Loans with principal of 0 or less:", (loans_clean[col] <= 0).sum())


In [ ]:
# How many active loans are there after removing duplicates (status 'active' in any letter case)?

In [ ]:

# 1. Find the status column automatically (or set status_col = 'status' directly)
status_col = [c for c in loans.columns if 'status' in c][0]

# 2. Remove duplicates (use subset=['loan_id'] to keep unique loans, or drop_duplicates() for exact rows)
loans_clean = loans.drop_duplicates(subset=['loan_id'])

# 3. Clean whitespace, convert to lowercase, and count rows matching 'active'
active_count = (loans_clean[status_col].astype(str).str.strip().str.lower() == 'active').sum()

print("Active loans after removing duplicates:", active_count)


In [ ]:
# How many active loans cannot be assessed because the gold weight is missing or the principal is 0 or less?

In [ ]:
# 1. Print exact columns first so you can see what is inside
print("Columns:", loans.columns.tolist())

# 2. Remove duplicate rows AND any accidental duplicate column names
loans_clean = loans.loc[:, ~loans.columns.duplicated()].drop_duplicates(subset=['loan_id']).copy()

# 3. Pick the exact single column names (change the strings below if your column names differ)
status_col = [c for c in loans_clean.columns if 'status' in c]
weight_col = [c for c in loans_clean.columns if 'weight' in c]
principal_col = [c for c in loans_clean.columns if 'principal' in c]

print("Using columns ->", status_col, "|", weight_col, "|", principal_col)

# 4. Clean each single Series safely (squeeze() forces a 1D Series)
s_status = loans_clean[status_col].squeeze().astype(str).str.strip().str.lower()

s_weight = pd.to_numeric(
    loans_clean[weight_col].squeeze().astype(str).str.replace(r'[^\d.-]', '', regex=True),
    errors='coerce'
)

s_principal = pd.to_numeric(
    loans_clean[principal_col].squeeze().astype(str).str.replace(r'[^\d.-]', '', regex=True),
    errors='coerce'
)

# 5. Filter: active loans where gold weight is missing OR principal <= 0
is_active = s_status == 'active'
missing_weight = s_weight.isna()
invalid_principal = s_principal <= 0

unassessable_count = (is_active & (missing_weight | invalid_principal)).sum()
print("Active loans that cannot be assessed:", unassessable_count)





In [ ]:
# How many daily rates are in gold_rates?

In [ ]:
# 1. Inspect columns and sample rows
print("Columns:", gold_rates.columns.tolist())
print(gold_rates.head(3))

# 2. Remove exact duplicate rows
gold_rates_clean = gold_rates.drop_duplicates()

# 3. Find the date column automatically
date_col = [c for c in gold_rates_clean.columns if 'date' in c]

print("Total rows (raw):", len(gold_rates))
print("Total rows (after removing exact duplicates):", len(gold_rates_clean))
print("Unique dates:", gold_rates_clean[date_col].nunique())


In [ ]:
import re
import pandas as pd


def normalize_karat(value):
  if value is None:
    return None
  text = str(value).strip().upper()

  hallmark_map = {"999": 24, "916": 22, "750": 18, "585": 14}
  if text in hallmark_map:
    return hallmark_map[text]

  match = re.search(r"\b(24|22|21|20|18|14|10|9)\b", text)
  if match:
    return int(match.group(1))
  return None


# Build DataFrame row-by-row so every value is clearly visible
rows = [
    (101, "22K", 50000),
    (102, "22 karat", 75000),
    (103, "18kt", 30000),
    (104, "916", 60000),
]

df = pd.DataFrame(rows, columns=["loan_id", "karat", "amount"])

# Create a clean integer column
df["clean_karat"] = df["karat"].apply(normalize_karat)

# 1. Filter for 22K loans
df_22k = df[df["clean_karat"] == 22]

# 2. Group by karat and calculate total loan amount per karat
summary = df.groupby("clean_karat")["amount"].sum()

print("22K Loans:\n", df_22k)
print("\nSummary by Karat:\n", summary)


In [ ]:
missing gold weight